# Designing a panel for secondary motor and prelimbic cortex  

Antonio Colás Nieto

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
import time
import matplotlib.pyplot as plt
import iss_analysis.io as io
import iss_analysis.pick_genes as pick

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

/camp/home/colasa/.conda/envs/rnaseq/lib/python3.10/site-packages/flexiznam/schema/sequencing_data.py:9: UserWarning: Could not find `sequencing_extensions` in config. Please update config file
  class SequencingData(Dataset):


KeyError: 'Author'

In [ ]:
download_base = Path('/nemo/lab/znamenskiyp/home/shared/projects/colasa_MOs_panel')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

## Main entry point example use case:

Use this only of you want to build the expression matrix from scratch

In [ ]:
df, gene_names = io.main_yao_2023(abc_cache, 
                  'WMB-10Xv3', 
                  ['WMB-10Xv3-Isocortex-2', 'WMB-10Xv3-Isocortex-1'], 
                  download_base,
                  region_of_interest = 'MO-FRP', 
                  neurotransmitters = ['GABA', 'Glut'])

In [4]:
datapath  =  '/nemo/lab/znamenskiyp/home/shared/projects/colasa_MOs_panel'
df, gene_names = io.read_yao_2023(datapath)

# Building expression matrix

In [ ]:
gene_names

In [ ]:
set(df['subclass'])

## Downloading, labelling and merging all Allen Brain data from MO-FRB (Somatomotor - Frontal pole)

Downloaded according to `abc_atlas_access`, we have two anndata files. If I understood Petr's stuff correctly, we'll need to use the raw counts files. 

In [ ]:
file = abc_cache.get_data_path(directory='WMB-10Xv3', file_name='WMB-10Xv3-Isocortex-2/raw')
print(file)
isocortex_2 = anndata.read_h5ad(file, backed='r')
print(isocortex_2)

file = abc_cache.get_data_path(directory='WMB-10Xv3', file_name='WMB-10Xv3-Isocortex-1/raw')
print(file)
isocortex_1 = anndata.read_h5ad(file, backed='r')
print(isocortex_1)

## Fetch metadata for region_of_interest and subclass ID

In [ ]:


directory = 'WMB-10Xv3'
filename_list = ['WMB-10Xv3-Isocortex-2', 'WMB-10Xv3-Isocortex-1']
region_of_interest = 'MO-FRP'
neurotransmitters = ['GABA', 'Glut']

filelist = []

for filename in filename_list:
    
    print('Downloading or accessing data. If not already in the cache, could be long')
    file = abc_cache.get_data_path(directory = directory, 
                                    file_name = f'{filename}/raw')
    
    expression_matrix = anndata.read_h5ad(file, backed='r')

    print('Building metadata')
    cell_extended = scratch.build_extended_metadata(abc_cache)

    print('Filtering data with roi/neurotransmitter')
    filtered_expression_matrix = expression_matrix_filter(expression_matrix, 
                                feature_matrix_label = filename, 
                                cell_extended = cell_extended,  
                                region_of_interest = region_of_interest, 
                                neurotransmitters = neurotransmitters)
    filelist.append(filtered_expression_matrix)

filelist = tuple(filelist)
assert filelist[0].var.equals(filelist[1].var), "The .var DataFrames are not identical."
#merging the aadata with 'unique' strategy preserves metadata in obs and var because there
#is only one possible value for it (a counterexample is the sum of all values for)
#a variable, or its mean, which change depending on the subset). In our case, 
#it's the gene names, so 'unique' for each variable. 
concat_expression_matrix = anndata.concat([filelist[0],filelist[1]], merge = 'unique')

In [26]:
assert filelist[0].var.equals(filelist[1].var), "The .var DataFrames are not identical."
concat_expression_matrix = anndata.concat([filelist[0],filelist[1]], merge = 'unique')

In [ ]:
concat_expression_matrix.var

In [9]:
filtered_expression_matrix = expression_matrix[(expression_matrix.obs['region_of_interest_acronym']==region_of_interest)]

In [ ]:
region_of_interest = 'MO-FRP'
sum(expression_matrix.obs['region_of_interest_acronym']==region_of_interest)

In [ ]:
filtered_expression_matrix

In [1]:
def expression_matrix_filter(expression_matrix, 
                             feature_matrix_label, 
                             cell_extended,  
                             region_of_interest = 'MO-FRP', 
                             neurotransmitters = ['GABA', 'Glut']):
    '''
    Generates an anndata object that has all of the metadata information for cluster and ROI. Because
    of how the anndata files are saved, this corresponds just to a feature_matrix. 
    I don't know why the number of entries for a feature matrix in the metadata
    is smaller than in the data by about 1K, but I've decided to not really care for
    now.  I've made a note and I'll probably raise an issue on their repo (scary)

    Args: 
        expression_matrix(anndata): a feature_matrix as read by abc_cache
        feature_matrix_label(str): a string indicating the name of the feature matrix on the server. 
        cell_extended(df): a pandas df generated by build_extended_metadata
        other params: filters for the kinds of cell you want to keep. 

    Out: 
        filtered_expression_matrix(anndata): an anndata object with massive metadata. 
    '''
    #Build metadata
    pred = (cell_extended['feature_matrix_label'] == feature_matrix_label)
    cell_filtered = cell_extended[pred]
    print('Cell labels in metadata:')
    print(len(cell_filtered.index))
    print('Cell labels in cells:')
    print(len(expression_matrix.obs.index))

    #Asserting labels are unique
    assert expression_matrix.obs.index.is_unique, "expression_matrix.obs has duplicate cell_label indices."
    assert cell_filtered.index.is_unique, "Metadata df has duplicate cell_label indices."

    #Dropping columns that are in both expression_matrix.obs and metadata
    #to allow joining of the two by cell label. 
    cell_filtered = cell_filtered.drop(columns = 'library_label')
    cell_filtered = cell_filtered.drop(columns = 'cell_barcode')

    #joining
    expression_matrix.obs = expression_matrix.obs.join(cell_filtered, how='left')

    print(expression_matrix)

    #filtering for the right ROI and neurotransmitter
    filtered_expression_matrix = expression_matrix[(expression_matrix.obs['region_of_interest_acronym']==region_of_interest) & (expression_matrix.obs['neurotransmitter'].isin(neurotransmitters))]

    print(filtered_expression_matrix)

    return filtered_expression_matrix


In [ ]:
print(expression_matrix)

We get the metadata dataframe, add on to it the roi information and cluster information, use that to subset isocortex 1 and 2

In [ ]:
cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)

In [ ]:
roi = abc_cache.get_metadata_dataframe(directory='WMB-10X', file_name='region_of_interest_metadata')
roi.set_index('acronym', inplace=True)
roi.rename(columns={'order': 'region_of_interest_order',
                    'color_hex_triplet': 'region_of_interest_color'},
           inplace=True)
roi.head(5)

In [ ]:
cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
)
cluster_details.set_index('cluster_alias', inplace=True)
cluster_details.head(5)

In [ ]:
cell_extended = cell.join(cluster_details, on='cluster_alias')
cell_extended = cell_extended.join(roi[['region_of_interest_order', 'region_of_interest_color']], on='region_of_interest_acronym')
cell_extended.head(5)

In [39]:
def expression_matrix_filter(expression_matrix, feature_matrix_label, cell_extended,  region_of_interest = 'MO-FRP', neurotransmitters = ['GABA', 'Glut']):

    #Build metadata
    pred = (cell_extended['feature_matrix_label'] == feature_matrix_label)
    cell_filtered = cell_extended[pred]
    print("Number of cells in metadata = ", len(cell_filtered))
    unique_cells = len(set(expression_matrix.obs['cell_barcode']))
    print(f'Number of cells in data = {len(expression_matrix)}')
    print(f'Unique barcodes in data = {unique_cells}')
    unique_barcodes = len(set(expression_matrix.obs.index))
    print(f'Unique cell labels in data = {unique_barcodes}')



    # Merge metadata with adata.obs, assuming metadata has 'cell_barcode' in the correct order
    merged_obs = cell_filtered[['cell_barcode']].merge(expression_matrix.obs, on='cell_barcode', how='left')

    # Update adata.obs to match the sorted order
    expression_matrix = expression_matrix[merged_obs.index, :]

    # Optionally, replace adata.obs with merged_obs if additional columns were added
    expression_matrix.obs = merged_obs

    filtered_expression_matrix = expression_matrix[(expression_matrix.obs['region_of_interest_acronym']==region_of_interest) & (expression_matrix.obs['neurotransmitter'].isin(neurotransmitters))]

    return filtered_expression_matrix

def build_extended_metadata(abc_cache, directory = 'WMB-10X'):
    #get basic metadata
    cell = abc_cache.get_metadata_dataframe(
    directory=directory,
    file_name='cell_metadata',
    dtype={'cell_label': str}
    )  
    cell.set_index('cell_label', inplace=True)

    #add region of interest
    roi = abc_cache.get_metadata_dataframe(directory=directory, file_name='region_of_interest_metadata')
    roi.set_index('acronym', inplace=True)
    roi.rename(columns={'order': 'region_of_interest_order',
                        'color_hex_triplet': 'region_of_interest_color'},
            inplace=True)
    
    #add clustering
    cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
    )
    cluster_details.set_index('cluster_alias', inplace=True)

    #merge
    cell_extended = cell.join(cluster_details, on='cluster_alias')
    cell_extended = cell_extended.join(roi[['region_of_interest_order', 'region_of_interest_color']], on='region_of_interest_acronym')

    return cell_extended

### Concatenating data and metadata

Their example use case. gene_filtered is just adata.var where the genes belong to a list of pre-specified genes. 

asubset is just adata indexed by gene_filtered. So this looks like the way to go: create a new obs that matches the metadata and then index the original anndata using the sorted obs. Can I do that?


In [ ]:
cell_extended = build_extended_metadata(abc_cache)

feature_matrix_label = ['WMB-10Xv3-Isocortex-2', 'WMB-10Xv3-Isocortex-1']

filtered_iso_2 = expression_matrix_filter(isocortex_2, feature_matrix_label[0], cell_extended)
filtered_iso_1 = expression_matrix_filter(isocortex_1, feature_matrix_label[1], cell_extended)

In [13]:
filtered_iso = anndata.concat((filtered_iso_1, filtered_iso_2))

In [ ]:
filtered_iso

In [19]:
# Extract the gene expression data (X matrix) and convert it to a DataFrame
df = pd.DataFrame(filtered_iso.X.toarray(), columns=filtered_iso.var_names)

# Add the 'subclass' and 'cluster' columns from the `obs` DataFrame to `df`
df['subclass'] = filtered_iso.obs['subclass']
df['cluster'] = filtered_iso.obs['cluster']

# Define the path to save the DataFrame as a CSV

csv_path = download_base / 'WMB_processed.csv'

df.to_csv(csv_path, index=False)




In [ ]:
print(f"X matrix size: {filtered_iso.X.data.nbytes / 1e6} MB")  # Memory usage of gene expression matrix
print(f"obs DataFrame size: {filtered_iso.obs.memory_usage(deep=True).sum() / 1e6} MB")
print(f"var DataFrame size: {filtered_iso.var.memory_usage(deep=True).sum() / 1e6} MB")
print(f"uns size (approx.): {sum([sys.getsizeof(v) for v in filtered_iso.uns.values()]) / 1e6} MB")


In [ ]:
filtered_iso.obs.head(5)

In [ ]:
pred = (cell_extended['feature_matrix_label'].isin(feature_matrix_label))
cell_filtered = cell_extended[pred]
cell_filtered.head(5)

# Running the pick_genes algorithm

In [2]:
savepath = '/nemo/lab/znamenskiyp/home/shared/projects/colasa_MOs_panel/panels'

In [3]:
pick.main(
    savepath,
    efficiency=0.01,
    datapath='/nemo/lab/znamenskiyp/home/shared/projects/colasa_MOs_panel',
    subsample=1,
    classify="cluster",
    dataset = 'tasic_2018'
)

NameError: name 'pick' is not defined